In [0]:
dbutils.library.restartPython()

In [0]:

%pip install   lightgbm
%pip install hyperopt
# Databricks notebook source
# Databricks notebook source
# Databricks notebook source
#

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # LightGBM Stock Close-Price Prediction Pipeline (v4 — Spark Feature Engineering + Sampled Hyperopt Tuning)
# MAGIC
# MAGIC **What changed vs. v3 (performance-only changes, logic unchanged):**
# MAGIC 1. **Feature engineering moved from Pandas to Spark.** In v3, all lag /
# MAGIC    rolling / volatility features were computed with
# MAGIC    `groupby("ticker").transform(lambda ...)` in Pandas *after*
# MAGIC    `toPandas()`. That calls a Python function separately for each of the
# MAGIC    ~13,550 ticker groups — on ~1.7M rows this is a severe bottleneck.
# MAGIC    We now compute every lag/rolling/target feature with native Spark
# MAGIC    window functions *before* collecting to pandas. Spark window
# MAGIC    functions are vectorized and distributed across the cluster, so this
# MAGIC    step drops from several minutes to seconds. `toPandas()` is still
# MAGIC    required afterward because LightGBM itself trains single-node, but
# MAGIC    now it's collecting already-finished features instead of raw
# MAGIC    columns that still need heavy per-group computation.
# MAGIC 2. **Hyperopt tuning now runs on a random sample of tickers**, not the
# MAGIC    full 1.7M-row train/val set. `SparkTrials` was deliberately NOT used
# MAGIC    to parallelize trials — two real problems rule it out here:
# MAGIC      a) `lgb.Dataset` holds C-level handles that aren't safely
# MAGIC         picklable across processes, so shipping the objective function
# MAGIC         (closing over `train_set`/`val_set`) to executors either fails
# MAGIC         serialization or forces rebuilding the Dataset (with its
# MAGIC         expensive feature binning) inside every task.
# MAGIC      b) LightGBM already parallelizes internally via threads on a
# MAGIC         single node. Running several `SparkTrials` concurrently without
# MAGIC         pinning `num_threads` per trial causes every trial to compete
# MAGIC         for the same cores, which can make tuning *slower*, not faster.
# MAGIC    Instead, `Trials()` stays sequential (unchanged from v3), but each
# MAGIC    trial now trains against a small, fixed random sample of tickers
# MAGIC    (`TUNING_SAMPLE_TICKERS`) so each trial is much cheaper. The time
# MAGIC    boundary is untouched: the sample is drawn from the same
# MAGIC    `train_pdf`/`val_pdf` produced by the Section 5 time-based split, so
# MAGIC    no trial ever sees data past the cutoff date. Once the best
# MAGIC    hyperparameters are found, the **final** model in Section 7 is
# MAGIC    retrained on the full `train_set`/`val_set` exactly as in v3 — the
# MAGIC    sampling only affects the search, never the deployed model.
# MAGIC
# MAGIC Everything else (holiday-aware sentiment join, ratio target, scale-free
# MAGIC features, atomic Gold write, NULL predictions for insufficient history,
# MAGIC scratch-table cleanup) is unchanged from v3.

# COMMAND ----------

# MAGIC %pip install lightgbm hyperopt

# COMMAND ----------

import numpy as np
import pandas as pd
import gc
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "finance_intelligence_hub"
LOOKBACK_MONTHS = 6              # matches step 2 below ("last 6 months")
LAG_DAYS = [1, 2, 3, 5, 10]       # lag features (replaces the LSTM's WINDOW_SIZE)
ROLLING_WINDOWS = [5, 10]        # rolling averages used as features
VOLATILITY_WINDOW = 10           # window for rolling volatility feature

GOLD_TABLE = f"{CATALOG}.gold.stock_price_predictions"
SCRATCH_SCHEMA = f"{CATALOG}.scratch"
TEMP_MERGED_TABLE = f"{SCRATCH_SCHEMA}.temp_lgbm_merged_inputs"
TEMP_FEATURED_TABLE = f"{SCRATCH_SCHEMA}.temp_lgbm_featured_inputs"

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. All companies (no Top-N limit)

# COMMAND ----------

all_tickers_df = spark.sql(f"""
    SELECT ticker
    FROM {CATALOG}.silver.companies
""")

top_tickers = [row.ticker for row in all_tickers_df.collect()]
print(f"✅ Number of companies selected: {len(top_tickers)}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Last 6 months of stock prices + daily sentiment (holiday-aware)
# MAGIC
# MAGIC Unchanged from v3: matching `pub_date == trade_date` directly drops any
# MAGIC news published on a day the market wasn't open (weekend / holiday) —
# MAGIC there's simply no `trade_date` row to match against, so that
# MAGIC sentiment vanishes before the `coalesce(0.0)` ever sees it. Instead, we
# MAGIC first build the distinct calendar of actual trading dates, then map
# MAGIC every `pub_date` forward to the *next trading date on or after it*
# MAGIC (a forward as-of match), and aggregate everything that lands on the
# MAGIC same trading date together.

# COMMAND ----------

six_months_ago = F.date_sub(F.current_date(), LOOKBACK_MONTHS * 30)

stock_prices_df = (
    spark.table(f"{CATALOG}.silver.stock_prices")
    .filter(F.col("ticker").isin(top_tickers))
    .filter(F.col("trade_date") >= six_months_ago)
    .select(
        "ticker", "trade_date", "open_price", "high_price",
        "low_price", "close_price", "adjusted_close_price", "volume"
    )
)

# Distinct trading calendar actually observed in the price data. This is the
# same calendar every ticker trades on, so it's independent of `ticker`.
trading_dates_df = stock_prices_df.select("trade_date").distinct()

raw_sentiment_df = (
    spark.table(f"{CATALOG}.silver.news_sentiments")
    .filter(F.col("ticker").isin(top_tickers))
    .withColumn("pub_date", F.to_date("published_at"))
    # keep sentiment only within a reasonable window matching the price
    # lookback, plus a small buffer so news right before the window's start
    # can still map forward correctly
    .filter(F.col("pub_date") >= F.date_sub(six_months_ago, 7))
)

# Forward as-of match: for every distinct pub_date, find the earliest
# trade_date that is >= pub_date. Uses a calendar spine + forward-fill
# (O(dates), not O(dates x trade_dates)) rather than a non-equi join.
all_pub_dates_df = raw_sentiment_df.select(F.col("pub_date").alias("cal_date")).distinct()
trading_dates_list = [row.trade_date for row in trading_dates_df.collect()]

calendar_spine_df = (
    trading_dates_df.select(F.col("trade_date").alias("cal_date"))
    .union(all_pub_dates_df)
    .distinct()
    .withColumn("is_trading_day", F.col("cal_date").isin(trading_dates_list))
)

w_desc = Window.orderBy(F.col("cal_date").desc()).rowsBetween(Window.unboundedPreceding, 0)

pub_to_trade_date_df = (
    calendar_spine_df
    .withColumn("trade_date_marker", F.when(F.col("is_trading_day"), F.col("cal_date")))
    .withColumn("mapped_trade_date", F.last("trade_date_marker", ignorenulls=True).over(w_desc))
    .select(F.col("cal_date").alias("pub_date"), "mapped_trade_date")
)
# Note: news published after the very last available trade_date (e.g. today,
# before tomorrow's session exists yet) gets a NULL mapped_trade_date and is
# dropped by the inner join below — there is no trading day yet for it to
# attach to.

daily_sentiment_df = (
    raw_sentiment_df.alias("raw")
    .join(
        pub_to_trade_date_df.alias("map"),
        F.col("raw.pub_date") == F.col("map.pub_date"),
        "inner",
    )
    .groupBy(F.col("raw.ticker").alias("ticker"), F.col("map.mapped_trade_date").alias("trade_date"))
    .agg(
        F.avg("positive_score").alias("avg_positive_score"),
        F.avg("negative_score").alias("avg_negative_score"),
        F.avg("neutral_score").alias("avg_neutral_score"),
        F.count("article_id").alias("news_count"),
    )
)

merged_df = (
    stock_prices_df.alias("sp")
    .join(
        daily_sentiment_df.alias("sent"),
        (F.col("sp.ticker") == F.col("sent.ticker"))
        & (F.col("sp.trade_date") == F.col("sent.trade_date")),
        "left",
    )
    .select(
        F.col("sp.ticker").alias("ticker"),
        F.col("sp.trade_date").alias("trade_date"),
        "open_price", "high_price", "low_price",
        "close_price", "adjusted_close_price", "volume",
        F.coalesce("avg_positive_score", F.lit(0.0)).alias("avg_positive_score"),
        F.coalesce("avg_negative_score", F.lit(0.0)).alias("avg_negative_score"),
        F.coalesce("avg_neutral_score", F.lit(0.0)).alias("avg_neutral_score"),
        F.coalesce("news_count", F.lit(0)).alias("news_count"),
    )
)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCRATCH_SCHEMA}")

(
    merged_df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(TEMP_MERGED_TABLE)
)

optimized_merged_df = spark.table(TEMP_MERGED_TABLE)
print(f"✅ Merged table saved temporarily at: {TEMP_MERGED_TABLE}")
print(f"✅ Merged rows: {optimized_merged_df.count()}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Feature engineering — IN SPARK (new in v4)
# MAGIC
# MAGIC Same feature definitions as v3, but computed with native Spark window
# MAGIC functions instead of `pandas.groupby(...).transform(lambda ...)`. This
# MAGIC is the single biggest win in this rewrite: the v3 Pandas version calls
# MAGIC a Python function per ticker group (~13,550 groups); this version is
# MAGIC vectorized and distributed across the cluster, independent of how many
# MAGIC tickers there are.
# MAGIC
# MAGIC All features remain scale-free, exactly matching v3's definitions:
# MAGIC - price/volume lags → log-return over each lag horizon
# MAGIC - rolling features → today's value relative to its own rolling mean
# MAGIC - close_pct_change / close_volatility → already scale-free
# MAGIC - target → log-return of next day's close vs. today's close

# COMMAND ----------

w_ticker_asc = Window.partitionBy("ticker").orderBy("trade_date")

featured_df = optimized_merged_df

feature_cols = []

# --- Scale-free price lags: log-return over each lag horizon
#
# NOTE: uses try_divide, not plain `/`. Under ANSI mode (Databricks default)
# a literal division by zero raises DIVIDE_BY_ZERO instead of returning NULL
# the way Pandas silently did in v3. A `close_price` of exactly 0 (bad data
# / a genuine data gap) makes its `lag(...)` value 0 too, which blows up the
# division before `log()` even runs. try_divide returns NULL for a zero
# denominator instead of erroring, matching v3's original (implicit) NaN
# behavior, and log(NULL) / log(<=0) both correctly propagate as NULL too.
price_cols = ["close_price", "adjusted_close_price"]
for col in price_cols:
    for lag in LAG_DAYS:
        fcol = f"{col}_logret_lag{lag}"
        featured_df = featured_df.withColumn(
            fcol,
            F.log(F.try_divide(F.col(col), F.lag(F.col(col), lag).over(w_ticker_asc)))
        )
        feature_cols.append(fcol)

# --- Scale-free volume lags: log-ratio vs. N days ago (+1 to avoid log(0))
# The +1 offsets make these denominators >= 1 as long as volume >= 0, so a
# true zero-division isn't expected here — try_divide is used anyway purely
# as a defensive guard against unexpected negative/NULL volume values.
for lag in LAG_DAYS:
    fcol = f"volume_logret_lag{lag}"
    featured_df = featured_df.withColumn(
        fcol,
        F.log(F.try_divide(F.col("volume") + 1, F.lag(F.col("volume"), lag).over(w_ticker_asc) + 1))
    )
    feature_cols.append(fcol)

# --- Sentiment lags: already bounded/comparable; news_count left raw as-is
sentiment_cols = ["avg_positive_score", "avg_negative_score", "avg_neutral_score", "news_count"]
for col in sentiment_cols:
    for lag in LAG_DAYS:
        fcol = f"{col}_lag{lag}"
        featured_df = featured_df.withColumn(fcol, F.lag(F.col(col), lag).over(w_ticker_asc))
        feature_cols.append(fcol)

# --- Scale-free rolling features: today's value relative to its own rolling mean
#
# Uses rangeBetween on trade_date (a DateType, which Spark's range frames
# treat as a day count), not rowsBetween. rowsBetween(-(w-1), 0) means
# "whatever the last w AVAILABLE rows are" — if a ticker was halted for
# several weeks, that window would silently reach back over the halt and
# average in stale pre-halt data alongside current data, understating how
# stale the signal actually is. rangeBetween(-(w-1), 0) instead means
# "whatever rows fall within the last w calendar days", so a long gap just
# yields fewer (or zero) rows in the window rather than quietly stretching
# back in time to fill a fixed row count.
for w in ROLLING_WINDOWS:
    # F.unix_date() converts the DateType trade_date to an integer day count
    # (days since epoch) so rangeBetween's numeric offsets are interpreted
    # as calendar days. A direct .cast("long") on a DateType column is not
    # a valid cast in Spark (DATE cannot be cast straight to a numeric type;
    # it must go through unix_date/datediff), so this is not just a style
    # choice — the .cast("long") form would fail outright.
    w_roll = Window.partitionBy("ticker").orderBy(F.unix_date(F.col("trade_date"))).rangeBetween(-(w - 1), 0)

    fcol = f"close_price_to_roll_mean{w}"
    featured_df = featured_df.withColumn(
        fcol, F.try_divide(F.col("close_price"), F.avg("close_price").over(w_roll)) - F.lit(1.0)
    )
    feature_cols.append(fcol)

    fcol2 = f"volume_to_roll_mean{w}"
    featured_df = featured_df.withColumn(
        fcol2,
        F.try_divide(F.col("volume") + 1, F.avg("volume").over(w_roll) + 1) - F.lit(1.0)
    )
    feature_cols.append(fcol2)

# --- Day-over-day pct change + rolling volatility of that pct change
# Same try_divide guard: yesterday's close being exactly 0 must not blow up
# the whole pipeline.
featured_df = featured_df.withColumn(
    "close_pct_change",
    F.try_divide(F.col("close_price"), F.lag("close_price", 1).over(w_ticker_asc)) - F.lit(1.0)
)
feature_cols.append("close_pct_change")

w_vol_roll = (
    Window.partitionBy("ticker")
    .orderBy(F.unix_date(F.col("trade_date")))
    .rangeBetween(-(VOLATILITY_WINDOW - 1), 0)
)
# Spark has no direct rolling-std convenience like pandas .rolling().std(),
# so compute sample stddev manually via stddev_samp over the same window —
# semantically identical to pandas' default (ddof=1) rolling std.
featured_df = featured_df.withColumn(
    "close_volatility",
    F.stddev_samp("close_pct_change").over(w_vol_roll)
)
feature_cols.append("close_volatility")

# --- Target: log-return of next day's ADJUSTED close relative to today's
# adjusted close (not raw close_price). Using raw close_price here would
# make a stock split or dividend distribution look like a real, catastrophic
# loss to the model: close_price drops mechanically on the ex-date, but
# shareholders haven't actually lost value. adjusted_close_price already
# factors splits/dividends out, so its log-return reflects the true economic
# return an investor experienced. This also means MAE/directional-accuracy
# in Section 9 aren't artificially wrecked by a split/dividend event landing
# inside the test window.
featured_df = featured_df.withColumn(
    "target_log_return",
    F.log(F.try_divide(
        F.lead("adjusted_close_price", 1).over(w_ticker_asc),
        F.col("adjusted_close_price"),
    ))
)

# Materialize once into scratch, same reasoning as the merged table: breaks
# lineage so the window-function DAG isn't recomputed on every downstream
# action, and keeps temp data out of curated silver.
(
    featured_df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(TEMP_FEATURED_TABLE)
)

optimized_featured_df = spark.table(TEMP_FEATURED_TABLE)
print(f"✅ Feature columns added ({len(feature_cols)}).")
print(f"✅ Featured rows: {optimized_featured_df.count()}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Collect into a single pandas DataFrame (global model)
# MAGIC
# MAGIC Unlike v3, this collects data that already has every feature computed
# MAGIC — Spark did the heavy per-ticker window computation, so this step is
# MAGIC just a data transfer, not a compute step.

# COMMAND ----------

featured_pdf = optimized_featured_df.orderBy("ticker", "trade_date").toPandas()
print(f"✅ Total rows collected to pandas: {len(featured_pdf)}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Time-based Train / Validation split (no shuffling, unchanged from v3)

# COMMAND ----------

# Last row per company = the row we'll generate the live prediction for (has
# no real target yet, since it represents "tomorrow")
last_rows = featured_pdf.groupby("ticker").tail(1).copy()

# Companies with too little trading history get skipped for prediction
# rather than fed fabricated/filled feature values.
last_rows_ready = last_rows.dropna(subset=feature_cols).copy()
last_rows_insufficient = last_rows.loc[
    last_rows.index.difference(last_rows_ready.index)
].copy()

if len(last_rows_insufficient) > 0:
    skipped_tickers = sorted(last_rows_insufficient["ticker"].unique().tolist())
    print(
        f"⚠️ Skipping predictions for {len(last_rows_insufficient)} companies "
        f"with insufficient trading history: {skipped_tickers}"
    )

# Remaining rows (which do have a real target) are the train/eval data —
# this drops each ticker's own warm-up period, not new-company rows
# specifically.
train_eval_pdf = featured_pdf.dropna(subset=feature_cols + ["target_log_return"]).copy()

# Time-based split anchored to the unique calendar of trading dates, not a
# row-count quantile. THREE-WAY split (train / val / test), not two-way:
#   - train: used to fit the model
#   - val:   used only for early stopping (Section 7) and Hyperopt tuning
#            (Section 6) — the model "sees" its own performance on this set
#            during training, so it cannot be used for an unbiased final
#            evaluation
#   - test:  never touched during training or tuning at all. Section 9's
#            evaluation runs against this set so the reported MAE/RMSE/
#            directional accuracy aren't inflated by the early-stopping
#            selection bias that comes from reusing val_pdf for evaluation.
unique_dates = sorted(train_eval_pdf["trade_date"].unique())
train_cutoff_idx = int(len(unique_dates) * 0.70)
val_cutoff_idx = int(len(unique_dates) * 0.85)

train_cutoff_date = unique_dates[train_cutoff_idx]
val_cutoff_date = unique_dates[val_cutoff_idx]

train_pdf = train_eval_pdf[train_eval_pdf["trade_date"] < train_cutoff_date].copy()
val_pdf = train_eval_pdf[
    (train_eval_pdf["trade_date"] >= train_cutoff_date)
    & (train_eval_pdf["trade_date"] < val_cutoff_date)
].copy()
test_pdf = train_eval_pdf[train_eval_pdf["trade_date"] >= val_cutoff_date].copy()

print(f"✅ Train rows: {len(train_pdf)} | Validation rows: {len(val_pdf)} | "
      f"Test rows: {len(test_pdf)} | "
      f"Predict rows: {len(last_rows_ready)} (+ {len(last_rows_insufficient)} skipped)")
print(f"✅ Train/val cutoff date: {train_cutoff_date} | Val/test cutoff date: {val_cutoff_date}")

# featured_pdf and train_eval_pdf were only staging steps to build the three
# splits + last_rows above; freeing them here keeps driver memory lower for
# the toPandas() collection of ~1.7M rows, especially once ticker counts
# grow beyond the current 13.6k.
del featured_pdf, train_eval_pdf, last_rows
gc.collect()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Hyperparameter tuning with Hyperopt — sampled tickers (changed in v4)
# MAGIC
# MAGIC Also starts the MLflow run for this pipeline execution (params, tuning
# MAGIC results, final model, and test-set metrics are all logged under the
# MAGIC same run — see `mlflow.start_run()` below and `mlflow.end_run()` at
# MAGIC the very end of Section 9).
# MAGIC
# MAGIC `Trials()` stays **sequential** — `SparkTrials` was deliberately not
# MAGIC adopted here. Two concrete reasons:
# MAGIC 1. `lgb.Dataset` wraps C-level handles that aren't safely picklable
# MAGIC    across processes, so shipping the objective closure to executors
# MAGIC    either breaks serialization or forces rebuilding the (expensive)
# MAGIC    binned Dataset inside every task.
# MAGIC 2. LightGBM already parallelizes internally via threads on a single
# MAGIC    node; running several `SparkTrials` concurrently without pinning
# MAGIC    `num_threads` per trial makes them compete for the same cores.
# MAGIC
# MAGIC Instead, the cost per trial is reduced directly: tuning trains only on
# MAGIC a **random sample of tickers** (`TUNING_SAMPLE_TICKERS`), drawn from
# MAGIC the same time-based `train_pdf`/`val_pdf` split — the sample changes
# MAGIC which companies are included, never which dates are, so no trial ever
# MAGIC sees data past the cutoff date. Once tuning finishes, the **final**
# MAGIC model in Section 7 is retrained on the FULL train/val data with the
# MAGIC winning hyperparameters — sampling only ever affects the search.

# COMMAND ----------

import lightgbm as lgb
import mlflow
import mlflow.lightgbm
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials

mlflow.autolog(disable=True)  # avoid duplicate/nested auto-logging from lgb.train inside the Hyperopt loop

EXPERIMENT_NAME = f"/Shared/{CATALOG}_lgbm_stock_prediction"
mlflow.set_experiment(EXPERIMENT_NAME)

# Safety net: a try/finally can't span separate notebook cells in Databricks
# (each cell executes as its own independent exec() call, so control flow
# can't carry over), which means we can't *guarantee* mlflow.end_run() runs
# if Section 6-9 throws partway through. Instead, guard the other direction:
# if a PREVIOUS failed execution left a run open (status stuck at
# "RUNNING"), close it here before starting a fresh one, so failures don't
# compound into orphaned runs piling up in the MLflow UI.
if mlflow.active_run() is not None:
    print(f"⚠️ Closing a leftover active MLflow run from a previous execution: "
          f"{mlflow.active_run().info.run_id}")
    mlflow.end_run(status="KILLED")

run = mlflow.start_run(run_name="lgbm_v4_spark_features_sampled_tuning")
mlflow.log_param("catalog", CATALOG)
mlflow.log_param("lookback_months", LOOKBACK_MONTHS)
mlflow.log_param("lag_days", str(LAG_DAYS))
mlflow.log_param("rolling_windows", str(ROLLING_WINDOWS))
mlflow.log_param("volatility_window", VOLATILITY_WINDOW)
mlflow.log_param("num_tickers", len(top_tickers))
mlflow.log_param("num_features", len(feature_cols))

train_pdf["ticker"] = train_pdf["ticker"].astype("category")
val_pdf["ticker"] = pd.Categorical(
    val_pdf["ticker"], categories=train_pdf["ticker"].cat.categories
)
test_pdf["ticker"] = pd.Categorical(
    test_pdf["ticker"], categories=train_pdf["ticker"].cat.categories
)
last_rows_ready = last_rows_ready.copy()
last_rows_ready["ticker"] = pd.Categorical(
    last_rows_ready["ticker"], categories=train_pdf["ticker"].cat.categories
)

lgbm_feature_cols = feature_cols + ["ticker"]

# Built once, reused by the final training run in Section 7. feature_pre_filter
# is disabled for consistency with tuning_train_set below — not strictly
# required here since this Dataset is only ever trained with one fixed
# min_data_in_leaf (best_params), but keeps behavior uniform and avoids
# surprises if this cell is re-run with different params later.
train_set = lgb.Dataset(
    train_pdf[lgbm_feature_cols], label=train_pdf["target_log_return"],
    categorical_feature=["ticker"],
    params={"feature_pre_filter": False},
)
val_set = lgb.Dataset(
    val_pdf[lgbm_feature_cols], label=val_pdf["target_log_return"],
    categorical_feature=["ticker"], reference=train_set,
)

# --- Sampled subset used ONLY for hyperparameter search ---
TUNING_SAMPLE_SIZE = min(1000, len(train_pdf["ticker"].cat.categories))
TUNING_SEED = 42

rng = np.random.RandomState(TUNING_SEED)
tuning_sample_tickers = rng.choice(
    train_pdf["ticker"].cat.categories.to_numpy(),
    size=TUNING_SAMPLE_SIZE,
    replace=False,
)

tuning_train_pdf = train_pdf[train_pdf["ticker"].isin(tuning_sample_tickers)]
tuning_val_pdf = val_pdf[val_pdf["ticker"].isin(tuning_sample_tickers)]

# Dataset built once for the sampled data, reused by every Hyperopt trial —
# same reasoning as v3: rebuilding per trial would multiply tuning time.
# feature_pre_filter=False is required here: LightGBM's Dataset defaults to
# pre-filtering features based on whichever min_data_in_leaf built the
# Dataset first, then errors if a later lgb.train() call passes a different
# min_data_in_leaf against that same Dataset. Since this Dataset is built
# once and reused across every Hyperopt trial (each proposing its own
# min_data_in_leaf), pre-filtering must be disabled or the second trial
# onward raises LightGBMError.
tuning_train_set = lgb.Dataset(
    tuning_train_pdf[lgbm_feature_cols], label=tuning_train_pdf["target_log_return"],
    categorical_feature=["ticker"],
    params={"feature_pre_filter": False},
)
tuning_val_set = lgb.Dataset(
    tuning_val_pdf[lgbm_feature_cols], label=tuning_val_pdf["target_log_return"],
    categorical_feature=["ticker"], reference=tuning_train_set,
)

print(f"✅ Tuning sample: {TUNING_SAMPLE_SIZE} tickers | "
      f"{len(tuning_train_pdf)} train rows | {len(tuning_val_pdf)} val rows "
      f"(full data has {len(train_pdf)} train / {len(val_pdf)} val rows)")

MAX_EVALS = 50

mlflow.log_param("train_rows", len(train_pdf))
mlflow.log_param("val_rows", len(val_pdf))
mlflow.log_param("test_rows", len(test_pdf))
mlflow.log_param("tuning_sample_size", TUNING_SAMPLE_SIZE)
mlflow.log_param("tuning_train_rows", len(tuning_train_pdf))
mlflow.log_param("tuning_val_rows", len(tuning_val_pdf))
mlflow.log_param("tuning_seed", TUNING_SEED)
mlflow.log_param("max_evals", MAX_EVALS)


def objective(space):
    params = {
        "objective": "regression",
        "metric": "mae",
        "learning_rate": space["learning_rate"],
        "num_leaves": int(space["num_leaves"]),
        "min_data_in_leaf": int(space["min_data_in_leaf"]),
        "feature_fraction": space["feature_fraction"],
        "bagging_fraction": space["bagging_fraction"],
        "bagging_freq": 1,
        "verbose": -1,
    }

    booster = lgb.train(
        params,
        tuning_train_set,
        num_boost_round=500,
        valid_sets=[tuning_train_set, tuning_val_set],
        valid_names=["train", "val"],
        callbacks=[
            lgb.early_stopping(stopping_rounds=30),
            lgb.log_evaluation(period=0),
        ],
    )

    val_mae = booster.best_score["val"]["l1"]
    return {
        "loss": val_mae,
        "status": STATUS_OK,
        "best_iteration": booster.best_iteration,
    }


search_space = {
    "learning_rate": hp.loguniform("learning_rate", np.log(0.005), np.log(0.2)),
    "num_leaves": hp.quniform("num_leaves", 15, 255, 1),
    "min_data_in_leaf": hp.quniform("min_data_in_leaf", 5, 100, 1),
    "feature_fraction": hp.uniform("feature_fraction", 0.5, 1.0),
    "bagging_fraction": hp.uniform("bagging_fraction", 0.5, 1.0),
}

trials = Trials()
best_raw_params = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=MAX_EVALS,
    trials=trials,
)

best_trial_result = trials.best_trial["result"]
best_num_boost_round = best_trial_result["best_iteration"]

best_params = {
    "objective": "regression",
    "metric": "mae",
    "learning_rate": best_raw_params["learning_rate"],
    # hp.quniform always returns a float (e.g. 63.0), even though LightGBM
    # requires these two as native ints and raises immediately if handed a
    # float. objective() casts them per-trial, but fmin's returned
    # best_raw_params are NOT re-cast automatically — this int() is what
    # protects the final training call in Section 7.
    "num_leaves": int(best_raw_params["num_leaves"]),
    "min_data_in_leaf": int(best_raw_params["min_data_in_leaf"]),
    "feature_fraction": best_raw_params["feature_fraction"],
    "bagging_fraction": best_raw_params["bagging_fraction"],
    "bagging_freq": 1,
    "verbose": -1,
}

print(f"✅ Best hyperparameters found after {MAX_EVALS} trials on the "
      f"{TUNING_SAMPLE_SIZE}-ticker tuning sample:")
print(best_params)
print(f"✅ Best iteration at best trial (on sample): {best_num_boost_round}")
print("ℹ️ Note: num_boost_round for the FINAL model (Section 7) is not taken "
      "from the sample's best_iteration — early stopping re-determines it "
      "against the full val_set, since the optimal round count on a sample "
      "of tickers doesn't necessarily transfer to the full dataset.")

mlflow.log_params({f"best_{k}": v for k, v in best_params.items() if k not in ("objective", "metric")})
mlflow.log_metric("tuning_sample_best_val_mae", best_trial_result["loss"])
mlflow.log_metric("tuning_sample_best_iteration", best_num_boost_round)

# The tuning sample and its Dataset objects are only needed during the
# search above — free them before the final full-data training run so they
# don't sit in driver memory alongside train_set/val_set unnecessarily.
del tuning_train_set, tuning_val_set, tuning_train_pdf, tuning_val_pdf, trials
gc.collect()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 7. Train the final LightGBM model on the FULL data with tuned hyperparameters
# MAGIC (unchanged from v3 — sampling in Section 6 only affected the search,
# MAGIC never this training call)

# COMMAND ----------

model = lgb.train(
    best_params,
    train_set,
    num_boost_round=500,
    valid_sets=[train_set, val_set],
    valid_names=["train", "val"],
    callbacks=[
        lgb.early_stopping(stopping_rounds=30),
        lgb.log_evaluation(period=0),
    ],
)

print(f"✅ Final training done with tuned hyperparameters on the FULL dataset. "
      f"Best iteration: {model.best_iteration}")

mlflow.log_metric("final_best_iteration", model.best_iteration)
mlflow.log_metric("final_train_mae", model.best_score["train"]["l1"])
mlflow.log_metric("final_val_mae", model.best_score["val"]["l1"])

# Log the trained booster as an MLflow model artifact. input_example helps
# MLflow infer/record a signature; ticker is already categorical here so the
# example reflects exactly what the model expects at inference time.
mlflow.lightgbm.log_model(
    model,
    artifact_path="lgbm_model",
    input_example=train_pdf[lgbm_feature_cols].head(5),
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 8. Final prediction — reconstruct absolute price + write to Gold atomically
# MAGIC (unchanged from v3)

# COMMAND ----------

predicted_log_return = model.predict(
    last_rows_ready[lgbm_feature_cols], num_iteration=model.best_iteration
)

last_rows_ready["predicted_log_return"] = predicted_log_return
# Reconstructed off adjusted_close_price, matching the target definition in
# Section 3 (log-return of adjusted close, not raw close). Using raw
# close_price here would silently reintroduce the split/dividend distortion
# the target was changed to avoid.
last_rows_ready["predicted_close_price"] = np.round(
    last_rows_ready["adjusted_close_price"] * np.exp(predicted_log_return), 2
)
last_rows_ready["last_known_close_price"] = last_rows_ready["adjusted_close_price"]
last_rows_ready["last_trade_date"] = last_rows_ready["trade_date"]
last_rows_ready["train_rows_used"] = float(len(train_pdf))

ready_predictions_pdf = last_rows_ready[[
    "ticker", "last_trade_date", "predicted_close_price",
    "predicted_log_return", "last_known_close_price", "train_rows_used",
]].copy()
ready_predictions_pdf["ticker"] = ready_predictions_pdf["ticker"].astype(str)
ready_predictions_pdf["last_trade_date"] = pd.to_datetime(ready_predictions_pdf["last_trade_date"])
for c in ["predicted_close_price", "predicted_log_return", "last_known_close_price", "train_rows_used"]:
    ready_predictions_pdf[c] = ready_predictions_pdf[c].astype("float64")

if len(last_rows_insufficient) > 0:
    insufficient_predictions_pdf = pd.DataFrame({
        "ticker": last_rows_insufficient["ticker"].astype(str).values,
        "last_trade_date": pd.to_datetime(last_rows_insufficient["trade_date"].values),
        "predicted_close_price": pd.array(
            [np.nan] * len(last_rows_insufficient), dtype="float64"
        ),
        "predicted_log_return": pd.array(
            [np.nan] * len(last_rows_insufficient), dtype="float64"
        ),
        "last_known_close_price": last_rows_insufficient["adjusted_close_price"].astype("float64").values,
        "train_rows_used": np.zeros(len(last_rows_insufficient), dtype="float64"),
    })
    predictions_pdf = pd.concat(
        [ready_predictions_pdf, insufficient_predictions_pdf], ignore_index=True
    )
else:
    predictions_pdf = ready_predictions_pdf.reset_index(drop=True)

predictions_pdf["ticker"] = predictions_pdf["ticker"].astype(str)
predictions_pdf["last_trade_date"] = pd.to_datetime(predictions_pdf["last_trade_date"])
predictions_pdf = predictions_pdf.astype({
    "predicted_close_price": "float64",
    "predicted_log_return": "float64",
    "last_known_close_price": "float64",
    "train_rows_used": "float64",
})

try:
    predictions_df = spark.createDataFrame(predictions_pdf)

    companies_df = spark.table(f"{CATALOG}.silver.companies")

    gold_df = (
        companies_df.alias("c")
        .join(predictions_df.alias("p"), "ticker", "inner")
        .select(
            "c.*",
            F.col("p.last_trade_date"),
            F.round(F.col("p.predicted_close_price"), 2).alias("predicted_close_price"),
            F.round(F.col("p.last_known_close_price"), 2).alias("last_known_close_price"),
            F.col("p.train_rows_used"),
            F.current_timestamp().alias("prediction_generated_at"),
        )
    )

    (
        gold_df.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable(GOLD_TABLE)
    )

    total_written = gold_df.count()
finally:
    # Clean up temporary tables even if the write above fails
    spark.sql(f"DROP TABLE IF EXISTS {TEMP_MERGED_TABLE}")
    spark.sql(f"DROP TABLE IF EXISTS {TEMP_FEATURED_TABLE}")

print(f"\n✅ Done! Predictions for {total_written} companies written to the Gold table: {GOLD_TABLE}")
display(spark.table(GOLD_TABLE).limit(20))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 9. Model Evaluation on Held-Out Test Set (Statistical & Financial)
# MAGIC
# MAGIC Evaluates against `test_pdf`, NOT `val_pdf`. `val_pdf` was used for
# MAGIC early stopping in Section 7 (and for Hyperopt tuning in Section 6) —
# MAGIC the model's training loop watched its own performance on `val_pdf` and
# MAGIC picked `best_iteration` based on it, so scoring against `val_pdf` again
# MAGIC here would give an optimistically biased read (the model was
# MAGIC implicitly selected to do well on exactly that data). `test_pdf` was
# MAGIC never touched during training or tuning, so it gives an honest,
# MAGIC unbiased estimate of real-world performance.

# COMMAND ----------

from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. Predict on the fully held-out test set
test_features = test_pdf[lgbm_feature_cols].copy()
test_pred_log_return = model.predict(test_features, num_iteration=model.best_iteration)

# 2. Reconstruct actual and predicted dollar prices
test_pdf_eval = test_pdf.copy()
test_pdf_eval["pred_log_return"] = test_pred_log_return

# Predicted / actual next-day price reconstructed off adjusted_close_price,
# matching the target definition in Section 3. Using raw close_price here
# would reintroduce split/dividend artifacts into the evaluation metrics —
# e.g. a stock split landing inside the test window would look like a huge
# MAE/directional-accuracy miss even though the model's return prediction
# was correct.
test_pdf_eval["pred_close_price"] = np.round(
    test_pdf_eval["adjusted_close_price"] * np.exp(test_pdf_eval["pred_log_return"]), 2
)

# Actual next-day adjusted close, reconstructed from the true
# target_log_return already stored in the featured data (Section 3, via
# F.lead() on adjusted_close_price).
test_pdf_eval["actual_next_close"] = np.round(
    test_pdf_eval["adjusted_close_price"] * np.exp(test_pdf_eval["target_log_return"]), 2
)

# --- Statistical metrics ---
mae = mean_absolute_error(test_pdf_eval["actual_next_close"], test_pdf_eval["pred_close_price"])
rmse = np.sqrt(mean_squared_error(test_pdf_eval["actual_next_close"], test_pdf_eval["pred_close_price"]))

# MAPE guard: a zero actual_next_close (bad data / halted ticker) would
# otherwise divide by zero and produce inf/NaN that silently poisons the
# mean. Those rows are excluded and reported separately instead.
mape_mask = test_pdf_eval["actual_next_close"] != 0
excluded_zero_actual = int((~mape_mask).sum())
mape = np.mean(
    np.abs(
        (test_pdf_eval.loc[mape_mask, "actual_next_close"] - test_pdf_eval.loc[mape_mask, "pred_close_price"])
        / test_pdf_eval.loc[mape_mask, "actual_next_close"]
    )
) * 100

print("==================================================")
print("📊 STATISTICAL EVALUATION (On Held-Out Test Set):")
print("==================================================")
print(f"🔹 Test rows: {len(test_pdf_eval)} | Date range: {test_pdf_eval['trade_date'].min()} to {test_pdf_eval['trade_date'].max()}")
print(f"🔹 Mean Absolute Error (MAE): ${mae:.2f}")
print(f"🔹 Root Mean Squared Error (RMSE): ${rmse:.2f}")
print(f"🔹 Mean Absolute Percentage Error (MAPE): {mape:.2f}%")
if excluded_zero_actual > 0:
    print(f"⚠️ Excluded {excluded_zero_actual} rows with zero actual_next_close from MAPE calculation")

# --- Directional accuracy ---
test_pdf_eval["actual_direction"] = (test_pdf_eval["target_log_return"] > 0).astype(int)
test_pdf_eval["pred_direction"] = (test_pdf_eval["pred_log_return"] > 0).astype(int)

directional_acc = (test_pdf_eval["actual_direction"] == test_pdf_eval["pred_direction"]).mean() * 100

print("\n==================================================")
print("📈 FINANCIAL DIRECTIONAL ACCURACY:")
print("==================================================")
print(f"🔹 Directional Accuracy: {directional_acc:.2f}%")
print("ℹ️ (In financial markets, a directional accuracy consistently above")
print("   ~53-55% on a genuinely held-out test set is considered a strong")
print("   trading signal. Numbers measured on the val_set used for early")
print("   stopping/tuning are not a reliable estimate — see Section 9 note.)")
print("==================================================")

# Final, unbiased metrics on the held-out test set — these are the numbers
# that should be trusted/compared across runs in the MLflow UI, not the
# tuning-sample or train/val metrics logged earlier.
mlflow.log_metric("test_mae", mae)
mlflow.log_metric("test_rmse", rmse)
mlflow.log_metric("test_mape", mape)
mlflow.log_metric("test_directional_accuracy", directional_acc)
mlflow.log_metric("test_rows", len(test_pdf_eval))
mlflow.log_metric("test_mape_excluded_zero_actual_rows", excluded_zero_actual)

mlflow.end_run()
print(f"✅ MLflow run logged: {run.info.run_id} (experiment: {EXPERIMENT_NAME})")

In [0]:
%sql
select min(published_at),max(published_at) from finance_intelligence_hub.gold.news_sentiments
